# Event Detection Validation AnalysisThis notebook validates the UPLIFT algorithm's automated event detection against human ground truth annotations for baseball hitting and pitching movements.**Contents:**- Tables 2-6: Summary statistics and reliability metrics- Figures 1-4: Visualizations of validation results

## 1. Setup & Data Loading

In [ ]:
# ============================================================================# IMPORTS AND CONFIGURATION# ============================================================================import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport pingouin as pgfrom sklearn.metrics import r2_score, mean_absolute_errorimport os# Set plot stylesns.set_style("whitegrid")plt.rcParams['figure.dpi'] = 100# UPLIFT brand colorUPLIFT_PINK = '#EC484F'# Frame rate conversion (240 fps)FRAME_TO_SEC = 1/240print("Libraries imported successfully")

In [ ]:
# ============================================================================# LOAD DATA FROM CSV FILES# ============================================================================# Load main comparison datahitting_df = pd.read_csv('../data/hitting_data.csv')pitching_df = pd.read_csv('../data/pitching_data.csv')# Load inter-rater datahitting_inter = pd.read_csv('../data/hitting_inter_rater.csv')pitching_inter = pd.read_csv('../data/pitching_inter_rater.csv')print(f"Data loaded successfully:")print(f"  Hitting trials: {len(hitting_df)} (Auto vs Human comparison)")print(f"  Pitching trials: {len(pitching_df)} (Auto vs Human comparison)")print(f"  Hitting inter-rater trials: {len(hitting_inter)} (Shun vs Ricky)")print(f"  Pitching inter-rater trials: {len(pitching_inter)} (Shun vs Ricky)")

In [ ]:
# ============================================================================# HELPER FUNCTIONS# ============================================================================def concordance_correlation_coefficient(y_true, y_pred):    """Calculate Lin's Concordance Correlation Coefficient (CCC)."""    y_true = np.array(y_true)    y_pred = np.array(y_pred)        mask = ~(np.isnan(y_true) | np.isnan(y_pred))    y_true, y_pred = y_true[mask], y_pred[mask]    n = len(y_true)        mean_true, mean_pred = np.mean(y_true), np.mean(y_pred)    var_true, var_pred = np.var(y_true, ddof=1), np.var(y_pred, ddof=1)    sd_true, sd_pred = np.std(y_true, ddof=1), np.std(y_pred, ddof=1)        covariance = np.cov(y_true, y_pred, ddof=1)[0, 1]    pearson_r = covariance / (sd_true * sd_pred)        v = sd_pred / sd_true    u = (mean_pred - mean_true) / np.sqrt(sd_pred * sd_true)    Cb = 2 / (v + 1/v + u**2)        ccc = pearson_r * Cb        # 95% CI using Fisher's z-transformation    z = 0.5 * np.log((1 + ccc) / (1 - ccc))    se_z = np.sqrt(1 / (n - 3))    z_lower, z_upper = z - 1.96 * se_z, z + 1.96 * se_z    ci_lower = (np.exp(2 * z_lower) - 1) / (np.exp(2 * z_lower) + 1)    ci_upper = (np.exp(2 * z_upper) - 1) / (np.exp(2 * z_upper) + 1)        return {'CCC': ccc, 'Pearson_r': pearson_r, 'Cb': Cb,             'CI95_lower': ci_lower, 'CI95_upper': ci_upper, 'n': n}def interpret_ccc(ccc):    if ccc >= 0.99: return "Almost Perfect"    elif ccc >= 0.95: return "Substantial"    elif ccc >= 0.90: return "Moderate"    else: return "Poor"print("Helper functions defined")

## 2. Tables

### Table 2: Error Distribution Statistics

In [ ]:
# ============================================================================# TABLE 2: ERROR DISTRIBUTION STATISTICS (Auto vs Human)# ============================================================================# Calculate differences (Human - Auto convention: positive = Auto detected EARLIER)# Hittinghit_fc_diff = hitting_df['Human_Foot_Contact'] - hitting_df['UPLIFT_Foot_Contact']hit_bc_diff = hitting_df['Human_Ball_Contact'] - hitting_df['UPLIFT_Ball_Contact']hit_st_diff = hitting_df['Human_Swing_Through'] - hitting_df['UPLIFT_Swing_Through']# Pitchingpitch_fc_diff = pitching_df['Human_Foot_Contact'] - pitching_df['UPLIFT_Foot_Contact']pitch_rel_diff = pitching_df['Human_Release'] - pitching_df['UPLIFT_Release']# Calculate R² valueshit_fc_r2 = r2_score(hitting_df['Human_Foot_Contact'], hitting_df['UPLIFT_Foot_Contact'])hit_bc_r2 = r2_score(hitting_df['Human_Ball_Contact'], hitting_df['UPLIFT_Ball_Contact'])hit_st_r2 = r2_score(hitting_df['Human_Swing_Through'], hitting_df['UPLIFT_Swing_Through'])pitch_fc_r2 = r2_score(pitching_df['Human_Foot_Contact'], pitching_df['UPLIFT_Foot_Contact'])pitch_rel_r2 = r2_score(pitching_df['Human_Release'], pitching_df['UPLIFT_Release'])# Calculate CCC valueshit_fc_ccc = concordance_correlation_coefficient(hitting_df['Human_Foot_Contact'], hitting_df['UPLIFT_Foot_Contact'])hit_bc_ccc = concordance_correlation_coefficient(hitting_df['Human_Ball_Contact'], hitting_df['UPLIFT_Ball_Contact'])hit_st_ccc = concordance_correlation_coefficient(hitting_df['Human_Swing_Through'], hitting_df['UPLIFT_Swing_Through'])pitch_fc_ccc = concordance_correlation_coefficient(pitching_df['Human_Foot_Contact'], pitching_df['UPLIFT_Foot_Contact'])pitch_rel_ccc = concordance_correlation_coefficient(pitching_df['Human_Release'], pitching_df['UPLIFT_Release'])# Create summary tabletable2_data = []for name, diff, r2, ccc in [    ('Hitting - Foot Contact', hit_fc_diff, hit_fc_r2, hit_fc_ccc),    ('Hitting - Ball Contact', hit_bc_diff, hit_bc_r2, hit_bc_ccc),    ('Hitting - Swing Through', hit_st_diff, hit_st_r2, hit_st_ccc),    ('Pitching - Foot Contact', pitch_fc_diff, pitch_fc_r2, pitch_fc_ccc),    ('Pitching - Release', pitch_rel_diff, pitch_rel_r2, pitch_rel_ccc),]:    table2_data.append({        'Event': name,        'N': len(diff),        'Mean (frames)': f"{diff.mean():.1f}",        'Mean (sec)': f"{diff.mean() * FRAME_TO_SEC:.3f}",        'SD (frames)': f"{diff.std():.1f}",        'Median': f"{diff.median():.0f}",        'MAE (frames)': f"{diff.abs().mean():.1f}",        'Range': f"[{diff.min():.0f}, {diff.max():.0f}]",        'R²': f"{r2:.3f}",        'CCC': f"{ccc['CCC']:.4f}"    })table2_df = pd.DataFrame(table2_data)print("TABLE 2: Error Distribution Statistics (Auto vs Human Ground Truth)")print("="*100)print(table2_df.to_string(index=False))

### Table 3: ICC Inter-Rater Reliability (Overview)

In [ ]:
# ============================================================================# TABLE 3: ICC INTER-RATER RELIABILITY (Shun vs Ricky)# ============================================================================# Prepare data for ICC calculation - long format required# Hittinghitting_icc_data = []for _, row in hitting_inter.iterrows():    hitting_icc_data.append({'ID': row['ID'], 'Rater': 'Shun',                              'Foot_Contact': row['Shun_Foot_Contact'],                             'Ball_Contact': row['Shun_Ball_Contact'],                             'Swing_Through': row['Shun_Swing_Through']})    hitting_icc_data.append({'ID': row['ID'], 'Rater': 'Ricky',                              'Foot_Contact': row['Ricky_Foot_Contact'],                             'Ball_Contact': row['Ricky_Ball_Contact'],                             'Swing_Through': row['Ricky_Swing_Through']})hitting_icc_df = pd.DataFrame(hitting_icc_data)# Pitchingpitching_icc_data = []for _, row in pitching_inter.iterrows():    pitching_icc_data.append({'ID': row['ID'], 'Rater': 'Shun',                              'Foot_Contact': row['Shun_Foot_Contact'],                              'Release': row['Shun_Release']})    pitching_icc_data.append({'ID': row['ID'], 'Rater': 'Ricky',                              'Foot_Contact': row['Ricky_Foot_Contact'],                              'Release': row['Ricky_Release']})pitching_icc_df = pd.DataFrame(pitching_icc_data)# Calculate ICC for each eventprint("TABLE 3: ICC Scores - Inter-Rater Reliability (Shun vs Ricky)")print("="*80)print("ICC Model: Two-way mixed-effects, absolute agreement (ICC3)")print()icc_results = []# Hitting eventsfor event in ['Foot_Contact', 'Ball_Contact', 'Swing_Through']:    icc = pg.intraclass_corr(data=hitting_icc_df, targets='ID', raters='Rater', ratings=event)    icc3 = icc[icc['Type'] == 'ICC3']['ICC'].values[0]    ci = icc[icc['Type'] == 'ICC3'][['CI95%']].values[0][0]    icc_results.append({'Movement': 'Hitting', 'Event': event.replace('_', ' '),                         'ICC': f"{icc3:.4f}", '95% CI': f"[{ci[0]:.4f}, {ci[1]:.4f}]",                        'N Trials': len(hitting_inter)})# Pitching eventsfor event in ['Foot_Contact', 'Release']:    icc = pg.intraclass_corr(data=pitching_icc_df, targets='ID', raters='Rater', ratings=event)    icc3 = icc[icc['Type'] == 'ICC3']['ICC'].values[0]    ci = icc[icc['Type'] == 'ICC3'][['CI95%']].values[0][0]    icc_results.append({'Movement': 'Pitching', 'Event': event.replace('_', ' '),                        'ICC': f"{icc3:.4f}", '95% CI': f"[{ci[0]:.4f}, {ci[1]:.4f}]",                        'N Trials': len(pitching_inter)})icc_table = pd.DataFrame(icc_results)print(icc_table.to_string(index=False))print()print("ICC Interpretation: >0.90 = Excellent, 0.75-0.90 = Good, 0.50-0.75 = Moderate, <0.50 = Poor")

### Table 4: ICC Scores (4 Decimal Places)

In [ ]:
# ============================================================================# TABLE 4: ICC VALUES WITH FULL PRECISION# ============================================================================# Store full precision valuesicc_full_precision = []# Hitting eventsfor event in ['Foot_Contact', 'Ball_Contact', 'Swing_Through']:    icc = pg.intraclass_corr(data=hitting_icc_df, targets='ID', raters='Rater', ratings=event)    icc3_row = icc[icc['Type'] == 'ICC3']    icc_val = icc3_row['ICC'].values[0]    ci = icc3_row['CI95%'].values[0]    pval = icc3_row['pval'].values[0]    icc_full_precision.append({        'Category': 'Hitting',         'Event': event.replace('_', ' '),        'ICC Score': f"{icc_val:.4f}",        '95% CI': f"[{ci[0]:.4f}, {ci[1]:.4f}]",        'p-value': '< 0.001' if pval < 0.001 else f"{pval:.4f}"    })# Pitching eventsfor event in ['Foot_Contact', 'Release']:    icc = pg.intraclass_corr(data=pitching_icc_df, targets='ID', raters='Rater', ratings=event)    icc3_row = icc[icc['Type'] == 'ICC3']    icc_val = icc3_row['ICC'].values[0]    ci = icc3_row['CI95%'].values[0]    pval = icc3_row['pval'].values[0]    icc_full_precision.append({        'Category': 'Pitching',        'Event': event.replace('_', ' '),        'ICC Score': f"{icc_val:.4f}",        '95% CI': f"[{ci[0]:.4f}, {ci[1]:.4f}]",        'p-value': '< 0.001' if pval < 0.001 else f"{pval:.4f}"    })table4_df = pd.DataFrame(icc_full_precision)print("TABLE 4: ICC Values (Inter-Rater Reliability)")print("="*80)print(table4_df.to_string(index=False))

### Table 5: CCC Auto vs Human

In [ ]:
# ============================================================================# TABLE 5: CCC VALUES (Auto vs Human Ground Truth)# ============================================================================table5_data = []for name, ccc_result in [    ('Hitting - Foot Contact', hit_fc_ccc),    ('Hitting - Ball Contact', hit_bc_ccc),    ('Hitting - Swing Through', hit_st_ccc),    ('Pitching - Foot Contact', pitch_fc_ccc),    ('Pitching - Release', pitch_rel_ccc),]:    movement, event = name.split(' - ')    table5_data.append({        'Movement': movement,        'Event': event,        'N': ccc_result['n'],        'CCC': f"{ccc_result['CCC']:.4f}",        'Pearson r': f"{ccc_result['Pearson_r']:.4f}",        'Bias (Cb)': f"{ccc_result['Cb']:.4f}",        '95% CI': f"[{ccc_result['CI95_lower']:.4f}, {ccc_result['CI95_upper']:.4f}]",        'Interpretation': interpret_ccc(ccc_result['CCC'])    })table5_df = pd.DataFrame(table5_data)print("TABLE 5: Concordance Correlation Coefficient (CCC) - Auto vs Human")print("="*100)print("CCC = Pearson_r × Cb (bias correction factor)")print()print(table5_df.to_string(index=False))

### Table 6: CCC Inter-Rater (Shun vs Ricky)

In [ ]:
# ============================================================================# TABLE 6: CCC VALUES (Inter-Rater: Shun vs Ricky)# ============================================================================table6_data = []# Hitting inter-rater CCCfor event in ['Foot_Contact', 'Ball_Contact', 'Swing_Through']:    shun_col = f'Shun_{event}'    ricky_col = f'Ricky_{event}'    ccc_result = concordance_correlation_coefficient(hitting_inter[shun_col], hitting_inter[ricky_col])    table6_data.append({        'Movement': 'Hitting',        'Event': event.replace('_', ' '),        'N': ccc_result['n'],        'CCC': f"{ccc_result['CCC']:.4f}",        'Pearson r': f"{ccc_result['Pearson_r']:.4f}",        'Bias (Cb)': f"{ccc_result['Cb']:.4f}",        '95% CI': f"[{ccc_result['CI95_lower']:.4f}, {ccc_result['CI95_upper']:.4f}]"    })# Pitching inter-rater CCCfor event in ['Foot_Contact', 'Release']:    shun_col = f'Shun_{event}'    ricky_col = f'Ricky_{event}'    ccc_result = concordance_correlation_coefficient(pitching_inter[shun_col], pitching_inter[ricky_col])    table6_data.append({        'Movement': 'Pitching',        'Event': event.replace('_', ' '),        'N': ccc_result['n'],        'CCC': f"{ccc_result['CCC']:.4f}",        'Pearson r': f"{ccc_result['Pearson_r']:.4f}",        'Bias (Cb)': f"{ccc_result['Cb']:.4f}",        '95% CI': f"[{ccc_result['CI95_lower']:.4f}, {ccc_result['CI95_upper']:.4f}]"    })table6_df = pd.DataFrame(table6_data)print("TABLE 6: CCC Values (Inter-Rater Reliability: Shun vs Ricky)")print("="*100)print(table6_df.to_string(index=False))

## 3. Figures

### Figure 1: Error Distribution Histograms

In [ ]:
# ============================================================================# FIGURE 1: ERROR DISTRIBUTION HISTOGRAMS# ============================================================================fig, axes = plt.subplots(3, 2, figsize=(14, 12))# Plot data: (differences, title, ax position)plots = [    (-hit_fc_diff, 'Hitting: Foot Contact', (0, 0)),    (-pitch_fc_diff, 'Pitching: Foot Contact', (0, 1)),    (-hit_bc_diff, 'Hitting: Ball Contact', (1, 0)),    (-pitch_rel_diff, 'Pitching: Release', (1, 1)),    (-hit_st_diff, 'Hitting: Swing Through', (2, 0)),]for diff, title, (row, col) in plots:    ax = axes[row, col]        # Histogram    ax.hist(diff, bins=30, color=UPLIFT_PINK, edgecolor='white', alpha=0.8)        # Add mean line    ax.axvline(diff.mean(), color='black', linestyle='--', linewidth=2, label=f'Mean: {diff.mean():.1f}')        # Labels    ax.set_xlabel('Error (Auto - Human) [frames]', fontsize=11)    ax.set_ylabel('Frequency', fontsize=11)    ax.set_title(f'{title}\n(n={len(diff)})', fontsize=12, fontweight='bold')    ax.legend(loc='upper right')# Hide empty subplotaxes[2, 1].axis('off')plt.suptitle('Figure 1: Error Distribution (Auto - Human)', fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.savefig('../figures/figure1_error_distribution.png', dpi=150, bbox_inches='tight')plt.show()

### Figure 2: Tolerance Interval ProportionsDistribution of detection accuracy across tolerance thresholds for **n=198 hitting trials** and **n=108 pitching trials**.

In [ ]:
# ============================================================================# FIGURE 2: TOLERANCE INTERVAL PROPORTIONS# ============================================================================def calculate_proportions(diff):    """Calculate proportion within tolerance thresholds."""    abs_diff = np.abs(diff)    n = len(abs_diff)    return {        'Very Accurate (0-6f)': (abs_diff <= 6).sum() / n * 100,        'Accurate (7-12f)': ((abs_diff > 6) & (abs_diff <= 12)).sum() / n * 100,        'Moderate (13-24f)': ((abs_diff > 12) & (abs_diff <= 24)).sum() / n * 100,        'Inaccurate (>24f)': (abs_diff > 24).sum() / n * 100,    }# Calculate for all eventsevents = ['Hitting:\nFoot Contact', 'Hitting:\nBall Contact', 'Hitting:\nSwing Through',          'Pitching:\nFoot Contact', 'Pitching:\nRelease']diffs = [hit_fc_diff, hit_bc_diff, hit_st_diff, pitch_fc_diff, pitch_rel_diff]props_data = {cat: [] for cat in ['Very Accurate (0-6f)', 'Accurate (7-12f)', 'Moderate (13-24f)', 'Inaccurate (>24f)']}for diff in diffs:    props = calculate_proportions(diff)    for cat, val in props.items():        props_data[cat].append(val)# Create stacked bar chartfig, ax = plt.subplots(figsize=(12, 6))x = np.arange(len(events))width = 0.6colors = ['#2E7D32', '#66BB6A', '#FFA726', '#EF5350']  # Green to redbottom = np.zeros(len(events))for i, (cat, values) in enumerate(props_data.items()):    bars = ax.bar(x, values, width, label=cat, bottom=bottom, color=colors[i])    # Add percentage labels    for j, (v, b) in enumerate(zip(values, bottom)):        if v > 5:  # Only label if > 5%            ax.text(j, b + v/2, f'{v:.0f}%', ha='center', va='center', fontsize=9, fontweight='bold')    bottom += valuesax.set_ylabel('Proportion (%)', fontsize=12)ax.set_xlabel('Event Type', fontsize=12)ax.set_title('Figure 2: Detection Accuracy by Tolerance Threshold', fontsize=14, fontweight='bold')ax.set_xticks(x)ax.set_xticklabels(events, fontsize=10)ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1))ax.set_ylim(0, 105)plt.tight_layout()plt.savefig('../figures/figure2_tolerance_intervals.png', dpi=150, bbox_inches='tight')plt.show()

### Figure 3: CCC Concordance Plots

In [ ]:
# ============================================================================# FIGURE 3: CCC CONCORDANCE PLOTS (Auto vs Human)# ============================================================================fig, axes = plt.subplots(3, 2, figsize=(14, 15))plots = [    (hitting_df['Human_Foot_Contact'], hitting_df['UPLIFT_Foot_Contact'], 'Hitting: Foot Contact', hit_fc_ccc, (0, 0)),    (pitching_df['Human_Foot_Contact'], pitching_df['UPLIFT_Foot_Contact'], 'Pitching: Foot Contact', pitch_fc_ccc, (0, 1)),    (hitting_df['Human_Ball_Contact'], hitting_df['UPLIFT_Ball_Contact'], 'Hitting: Ball Contact', hit_bc_ccc, (1, 0)),    (pitching_df['Human_Release'], pitching_df['UPLIFT_Release'], 'Pitching: Release', pitch_rel_ccc, (1, 1)),    (hitting_df['Human_Swing_Through'], hitting_df['UPLIFT_Swing_Through'], 'Hitting: Swing Through', hit_st_ccc, (2, 0)),]for human, auto, title, ccc_result, (row, col) in plots:    ax = axes[row, col]        # Scatter plot    ax.scatter(human, auto, alpha=0.6, s=50, c='#4C72B0', edgecolors='white', linewidth=0.5)        # Line of perfect concordance    all_vals = np.concatenate([human.values, auto.values])    min_val, max_val = all_vals.min() - 10, all_vals.max() + 10    ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2, label='Perfect Concordance')        # Labels    ax.set_xlabel('Human Annotation (frames)', fontsize=11)    ax.set_ylabel('UPLIFT Detection (frames)', fontsize=11)    ax.set_title(f"{title}\nCCC = {ccc_result['CCC']:.4f}", fontsize=12, fontweight='bold')    ax.legend(loc='upper left')    ax.set_xlim(min_val, max_val)    ax.set_ylim(min_val, max_val)# Hide empty subplotaxes[2, 1].axis('off')plt.suptitle('Figure 3: CCC Concordance Plots (Auto vs Human)', fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.savefig('../figures/figure3_ccc_concordance.png', dpi=150, bbox_inches='tight')plt.show()

### Figure 4: Bland-Altman Plots

In [ ]:
# ============================================================================# FIGURE 4: BLAND-ALTMAN PLOTS (Inter-Rater: Shun vs Ricky)# ============================================================================fig, axes = plt.subplots(2, 3, figsize=(15, 10))# Hitting eventshitting_events = [('Foot_Contact', 'Foot Contact'), ('Ball_Contact', 'Ball Contact'), ('Swing_Through', 'Swing Through')]for i, (col_name, display_name) in enumerate(hitting_events):    ax = axes[0, i]    shun = hitting_inter[f'Shun_{col_name}']    ricky = hitting_inter[f'Ricky_{col_name}']        mean_vals = (shun + ricky) / 2    diff_vals = shun - ricky        ax.scatter(mean_vals, diff_vals, alpha=0.7, s=60, c=UPLIFT_PINK, edgecolors='white')        # Mean and limits of agreement    mean_diff = diff_vals.mean()    sd_diff = diff_vals.std()    ax.axhline(mean_diff, color='blue', linestyle='-', linewidth=2, label=f'Mean: {mean_diff:.2f}')    ax.axhline(mean_diff + 1.96*sd_diff, color='red', linestyle='--', linewidth=1.5, label=f'+1.96 SD: {mean_diff + 1.96*sd_diff:.2f}')    ax.axhline(mean_diff - 1.96*sd_diff, color='red', linestyle='--', linewidth=1.5, label=f'-1.96 SD: {mean_diff - 1.96*sd_diff:.2f}')    ax.axhline(0, color='gray', linestyle=':', linewidth=1)        ax.set_xlabel('Mean (Shun + Ricky) / 2', fontsize=10)    ax.set_ylabel('Difference (Shun - Ricky)', fontsize=10)    ax.set_title(f'Hitting: {display_name}\n(n={len(diff_vals)})', fontsize=11, fontweight='bold')    ax.legend(fontsize=8, loc='upper right')# Pitching eventspitching_events = [('Foot_Contact', 'Foot Contact'), ('Release', 'Release')]for i, (col_name, display_name) in enumerate(pitching_events):    ax = axes[1, i]    shun = pitching_inter[f'Shun_{col_name}']    ricky = pitching_inter[f'Ricky_{col_name}']        mean_vals = (shun + ricky) / 2    diff_vals = shun - ricky        ax.scatter(mean_vals, diff_vals, alpha=0.7, s=60, c=UPLIFT_PINK, edgecolors='white')        mean_diff = diff_vals.mean()    sd_diff = diff_vals.std()    ax.axhline(mean_diff, color='blue', linestyle='-', linewidth=2, label=f'Mean: {mean_diff:.2f}')    ax.axhline(mean_diff + 1.96*sd_diff, color='red', linestyle='--', linewidth=1.5, label=f'+1.96 SD: {mean_diff + 1.96*sd_diff:.2f}')    ax.axhline(mean_diff - 1.96*sd_diff, color='red', linestyle='--', linewidth=1.5, label=f'-1.96 SD: {mean_diff - 1.96*sd_diff:.2f}')    ax.axhline(0, color='gray', linestyle=':', linewidth=1)        ax.set_xlabel('Mean (Shun + Ricky) / 2', fontsize=10)    ax.set_ylabel('Difference (Shun - Ricky)', fontsize=10)    ax.set_title(f'Pitching: {display_name}\n(n={len(diff_vals)})', fontsize=11, fontweight='bold')    ax.legend(fontsize=8, loc='upper right')# Hide empty subplotaxes[1, 2].axis('off')plt.suptitle('Figure 4: Bland-Altman Plots (Inter-Rater Agreement)', fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.savefig('../figures/figure4_bland_altman.png', dpi=150, bbox_inches='tight')plt.show()

## SummaryAll analyses complete. Key findings:- **CCC (Auto vs Human)**: All events show CCC > 0.99 (Almost Perfect agreement)- **ICC (Inter-Rater)**: All events show ICC > 0.99 (Excellent reliability)- **Error Distribution**: Mean errors within ±5 frames (±21 ms at 240 fps)- **Tolerance Analysis**: >90% of detections within ±12 frames of ground truth